# CNN-GRU Preprocessing
Downsamples to 20 Hz per paper, extracts 5s windows across all datasets, and saves to cnn_gru/data/windowed/

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import resample_poly, butter, filtfilt

sf_dict = {"fog_star": 60.0, "omnia_park": 90.0, "pd_phone": 200.0, "wearpd": 100.0, "kiel": 200.0}
datasets = ["fog_star", "omnia_park", "pd_phone", "wearpd", "kiel"]
target_hz = 20.0

OUT_PRE = Path('data') / 'preprocessed_data'
OUT_WIN = Path('data') / 'windowed'
OUT_PRE.mkdir(parents=True, exist_ok=True)
OUT_WIN.mkdir(parents=True, exist_ok=True)

print(f'Target sampling rate: {target_hz} Hz')

In [ ]:
def enforce_nan_policy(group, sf, max_interp_gap_sec=0.2, sensor_cols=None):
    if sensor_cols is None:
        sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    group = group.sort_values('timestamp').copy()
    max_interp_gap = max(1, int(max_interp_gap_sec * sf))
    for col in sensor_cols:
        isna = group[col].isna().to_numpy()
        if isna.any():
            edges = np.diff(np.r_[False, isna, False].astype(int))
            starts = np.where(edges == 1)[0]
            ends = np.where(edges == -1)[0]
            if len(ends) > 0:
                max_gap = int((ends - starts).max())
                if max_gap > max_interp_gap:
                    return None
            group[col] = group[col].interpolate(method='linear', limit_direction='both')
        if group[col].isna().any():
            return None
    return group

def trim_outliers(group, sf, z_threshold=2.5, trim_perc=0.15, window_size_sec=3.0):
    group = group.sort_values('timestamp').copy()
    total_duration = group['timestamp'].max() - group['timestamp'].min()
    n_trim = int(trim_perc * total_duration * sf)
    if len(group) < (2 * n_trim + sf):
        return None
    group = group.iloc[n_trim:-n_trim].copy()
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    window_size = max(3, int(window_size_sec * sf))
    for col in sensor_cols:
        rolling = group[col].rolling(window=window_size, center=True, min_periods=1)
        z_score = np.abs((group[col] - rolling.mean()) / (rolling.std() + 1e-6))
        group.loc[z_score > z_threshold, col] = np.nan
    group = enforce_nan_policy(group, sf, max_interp_gap_sec=0.2, sensor_cols=sensor_cols)
    if group is None:
        return None
    group['timestamp'] = (group['timestamp'] - group['timestamp'].min()).round(4)
    return group

def lowpass_filter(group, sf, cutoff=15.0, order=4):
    nyquist = 0.5 * sf
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    for col in sensor_cols:
        mean_val = group[col].mean()
        signal_centered = group[col].values - mean_val
        filtered_centered = filtfilt(b, a, signal_centered)
        group[col] = filtered_centered + mean_val
    return group

def resample(group, original_sf, target_sf=20.0):
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    up, down = int(target_sf), int(original_sf)
    pad_samples = int(original_sf)
    n_orig = len(group)
    n_target = int(n_orig * target_sf / original_sf)
    resampled_data = {
        'timestamp': np.linspace(0, (n_orig-1)/original_sf, n_target),
        'subjectID': group['subjectID'].iloc[0],
        'sessionID': group['sessionID'].iloc[0],
        'taskID': group['taskID'].iloc[0]
    }
    for col in sensor_cols:
        padded = np.pad(group[col].values, pad_width=pad_samples, mode='reflect')
        resampled_padded = resample_poly(padded, up, down)
        pad_target = int(pad_samples * target_sf / original_sf)
        resampled_data[col] = resampled_padded[pad_target : pad_target + n_target]
    return pd.DataFrame(resampled_data)

print('Preprocessing functions defined')

In [ ]:
os.makedirs(OUT_PRE, exist_ok=True)
sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

for dataset in datasets:
    path = f"../data/cleaned_data/{dataset}_sensor.csv"
    if not os.path.exists(path):
        print(f'Skipping {dataset} (not found)')
        continue

    print(f'\n--- Processing {dataset} ---')
    df = pd.read_csv(path)
    sf = sf_dict[dataset]

    df_filtered = df[df.taskID.isin([0, 1, 2])].copy()
    processed_sessions = []
    total_count, discarded_count = 0, 0
    grouped = df_filtered.groupby(['subjectID', 'sessionID', 'taskID'])

    for _, session_group in grouped:
        total_count += 1
        if session_group['taskID'].iloc[0] in [0, 1]:
            temp_group = trim_outliers(session_group, sf)
            if temp_group is None:
                discarded_count += 1
                continue
        else:
            temp_group = session_group.sort_values('timestamp').iloc[int(sf):-int(sf)].copy()
            if len(temp_group) <= sf:
                discarded_count += 1
                continue
            temp_group = enforce_nan_policy(temp_group, sf, max_interp_gap_sec=0.2, sensor_cols=sensor_cols)
            if temp_group is None:
                discarded_count += 1
                continue
            temp_group['timestamp'] = (temp_group['timestamp'] - temp_group['timestamp'].min()).round(4)

        temp_group = lowpass_filter(temp_group, sf, cutoff=15.0)
        temp_group = resample(temp_group, sf, target_sf=target_hz)

        if temp_group[sensor_cols].isna().any().any():
            discarded_count += 1
            continue

        processed_sessions.append(temp_group)

    if processed_sessions:
        out_df = pd.concat(processed_sessions, ignore_index=True)
        valid_mask = (
            out_df.groupby(['subjectID', 'sessionID', 'taskID'])[sensor_cols]
            .transform(lambda x: ~x.isna().any())
            .all(axis=1)
        )
        out_df = out_df.loc[valid_mask].copy()
        out_df = out_df.dropna(subset=sensor_cols)
        out_df.to_csv(OUT_PRE / f"{dataset}_sensor.csv", index=False)
        print(f'Saved: {total_count - discarded_count}/{total_count} sessions')
    else:
        print(f'No valid sessions for {dataset}')

In [ ]:
all_dfs = []
for dataset in datasets:
    path = OUT_PRE / f"{dataset}_sensor.csv"
    if path.exists():
        temp_df = pd.read_csv(path)
        temp_df['dataset'] = dataset
        all_dfs.append(temp_df)

if not all_dfs:
    print('No preprocessed datasets found')
else:
    full_df = pd.concat(all_dfs, ignore_index=True)
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    window_size = int(target_hz * 5)
    overlap = 0.75
    step = int(window_size * (1 - overlap))

    X = []
    meta = []
    grouped = full_df.groupby(['subjectID', 'sessionID', 'taskID', 'dataset'])
    for (sub_id, sess_id, task_id, dataset), group in grouped:
        data = group[sensor_cols].values.astype(np.float32)
        if len(data) < window_size:
            continue
        for i in range(0, len(data) - window_size + 1, step):
            X.append(data[i : i + window_size])
            meta.append({
                'subjectID': sub_id,
                'sessionID': sess_id,
                'taskID': task_id,
                'dataset': dataset
            })

    X = np.array(X, dtype=np.float32)
    meta_df = pd.DataFrame(meta)

    rng = np.random.default_rng(42)
    perm = rng.permutation(len(X))
    X = X[perm]
    meta_df = meta_df.iloc[perm].reset_index(drop=True)

    out_dir = Path('.') / 'data' / 'windowed'
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'windows.npy', X)
    meta_df.to_csv(out_dir / 'metadata.csv', index=False)
    print(f'Wrote {len(X)} windows to {out_dir}')
    print(f'Window shape: {X.shape}')

# CNN-GRU Preprocessing
Downsamples to 20 Hz per paper, extracts 5s windows across all datasets, and saves to cnn_gru/data/windowed/

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import resample_poly, butter, filtfilt

sf_dict = {"fog_star": 60.0, "omnia_park": 90.0, "pd_phone": 200.0, "wearpd": 100.0, "kiel": 200.0}
datasets = ["fog_star", "omnia_park", "pd_phone", "wearpd", "kiel"]
target_hz = 20.0

OUT_PRE = Path('data') / 'preprocessed_data'
OUT_WIN = Path('data') / 'windowed'
OUT_PRE.mkdir(parents=True, exist_ok=True)
OUT_WIN.mkdir(parents=True, exist_ok=True)

print(f'Target sampling rate: {target_hz} Hz')

In [ ]:
def enforce_nan_policy(group, sf, max_interp_gap_sec=0.2, sensor_cols=None):
    if sensor_cols is None:
        sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    group = group.sort_values('timestamp').copy()
    max_interp_gap = max(1, int(max_interp_gap_sec * sf))
    for col in sensor_cols:
        isna = group[col].isna().to_numpy()
        if isna.any():
            edges = np.diff(np.r_[False, isna, False].astype(int))
            starts = np.where(edges == 1)[0]
            ends = np.where(edges == -1)[0]
            if len(ends) > 0:
                max_gap = int((ends - starts).max())
                if max_gap > max_interp_gap:
                    return None
            group[col] = group[col].interpolate(method='linear', limit_direction='both')
        if group[col].isna().any():
            return None
    return group

def trim_outliers(group, sf, z_threshold=2.5, trim_perc=0.15, window_size_sec=3.0):
    group = group.sort_values('timestamp').copy()
    total_duration = group['timestamp'].max() - group['timestamp'].min()
    n_trim = int(trim_perc * total_duration * sf)
    if len(group) < (2 * n_trim + sf):
        return None
    group = group.iloc[n_trim:-n_trim].copy()
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    window_size = max(3, int(window_size_sec * sf))
    for col in sensor_cols:
        rolling = group[col].rolling(window=window_size, center=True, min_periods=1)
        z_score = np.abs((group[col] - rolling.mean()) / (rolling.std() + 1e-6))
        group.loc[z_score > z_threshold, col] = np.nan
    group = enforce_nan_policy(group, sf, max_interp_gap_sec=0.2, sensor_cols=sensor_cols)
    if group is None:
        return None
    group['timestamp'] = (group['timestamp'] - group['timestamp'].min()).round(4)
    return group

def lowpass_filter(group, sf, cutoff=15.0, order=4):
    nyquist = 0.5 * sf
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    for col in sensor_cols:
        mean_val = group[col].mean()
        signal_centered = group[col].values - mean_val
        filtered_centered = filtfilt(b, a, signal_centered)
        group[col] = filtered_centered + mean_val
    return group

def resample(group, original_sf, target_sf=20.0):
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    up, down = int(target_sf), int(original_sf)
    pad_samples = int(original_sf)
    n_orig = len(group)
    n_target = int(n_orig * target_sf / original_sf)
    resampled_data = {
        'timestamp': np.linspace(0, (n_orig-1)/original_sf, n_target),
        'subjectID': group['subjectID'].iloc[0],
        'sessionID': group['sessionID'].iloc[0],
        'taskID': group['taskID'].iloc[0]
    }
    for col in sensor_cols:
        padded = np.pad(group[col].values, pad_width=pad_samples, mode='reflect')
        resampled_padded = resample_poly(padded, up, down)
        pad_target = int(pad_samples * target_sf / original_sf)
        resampled_data[col] = resampled_padded[pad_target : pad_target + n_target]
    return pd.DataFrame(resampled_data)

print('Preprocessing functions defined')

In [ ]:
os.makedirs(OUT_PRE, exist_ok=True)
sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

for dataset in datasets:
    path = f"../data/cleaned_data/{dataset}_sensor.csv"
    if not os.path.exists(path):
        print(f'Skipping {dataset} (not found)')
        continue

    print(f'\n--- Processing {dataset} ---')
    df = pd.read_csv(path)
    sf = sf_dict[dataset]

    df_filtered = df[df.taskID.isin([0, 1, 2])].copy()
    processed_sessions = []
    total_count, discarded_count = 0, 0
    grouped = df_filtered.groupby(['subjectID', 'sessionID', 'taskID'])

    for _, session_group in grouped:
        total_count += 1
        if session_group['taskID'].iloc[0] in [0, 1]:
            temp_group = trim_outliers(session_group, sf)
            if temp_group is None:
                discarded_count += 1
                continue
        else:
            temp_group = session_group.sort_values('timestamp').iloc[int(sf):-int(sf)].copy()
            if len(temp_group) <= sf:
                discarded_count += 1
                continue
            temp_group = enforce_nan_policy(temp_group, sf, max_interp_gap_sec=0.2, sensor_cols=sensor_cols)
            if temp_group is None:
                discarded_count += 1
                continue
            temp_group['timestamp'] = (temp_group['timestamp'] - temp_group['timestamp'].min()).round(4)

        temp_group = lowpass_filter(temp_group, sf, cutoff=15.0)
        temp_group = resample(temp_group, sf, target_sf=target_hz)

        if temp_group[sensor_cols].isna().any().any():
            discarded_count += 1
            continue

        processed_sessions.append(temp_group)

    if processed_sessions:
        out_df = pd.concat(processed_sessions, ignore_index=True)
        valid_mask = (
            out_df.groupby(['subjectID', 'sessionID', 'taskID'])[sensor_cols]
            .transform(lambda x: ~x.isna().any())
            .all(axis=1)
        )
        out_df = out_df.loc[valid_mask].copy()
        out_df = out_df.dropna(subset=sensor_cols)
        out_df.to_csv(OUT_PRE / f"{dataset}_sensor.csv", index=False)
        print(f'Saved: {total_count - discarded_count}/{total_count} sessions')
    else:
        print(f'No valid sessions for {dataset}')

In [ ]:
all_dfs = []
for dataset in datasets:
    path = OUT_PRE / f"{dataset}_sensor.csv"
    if path.exists():
        temp_df = pd.read_csv(path)
        temp_df['dataset'] = dataset
        all_dfs.append(temp_df)

if not all_dfs:
    print('No preprocessed datasets found')
else:
    full_df = pd.concat(all_dfs, ignore_index=True)
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    window_size = int(target_hz * 5)
    overlap = 0.75
    step = int(window_size * (1 - overlap))

    X = []
    meta = []
    grouped = full_df.groupby(['subjectID', 'sessionID', 'taskID', 'dataset'])
    for (sub_id, sess_id, task_id, dataset), group in grouped:
        data = group[sensor_cols].values.astype(np.float32)
        if len(data) < window_size:
            continue
        for i in range(0, len(data) - window_size + 1, step):
            X.append(data[i : i + window_size])
            meta.append({
                'subjectID': sub_id,
                'sessionID': sess_id,
                'taskID': task_id,
                'dataset': dataset
            })

    X = np.array(X, dtype=np.float32)
    meta_df = pd.DataFrame(meta)

    rng = np.random.default_rng(42)
    perm = rng.permutation(len(X))
    X = X[perm]
    meta_df = meta_df.iloc[perm].reset_index(drop=True)

    out_dir = Path('.') / 'data' / 'windowed'
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'windows.npy', X)
    meta_df.to_csv(out_dir / 'metadata.csv', index=False)
    print(f'Wrote {len(X)} windows to {out_dir}')
    print(f'Window shape: {X.shape}')
    print(f'Class distribution: {meta_df.value_counts()}')